In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python langchain-tavily rapidfuzz langgraph exa-py serpapi google-search-results

In [2]:
import os, getpass
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch, TavilyExtract
from typing import Any, Dict
from typing_extensions import TypedDict, NotRequired, Literal
from exa_py import Exa

# Load environment variables from .env file
load_dotenv(dotenv_path="etl/.env")

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
_set_env("LANGCHAIN_API_KEY")
_set_env("TAVILY_API_KEY")
_set_env("EXA_API_KEY")
_set_env("BING_SUBSCRIPTION_KEY")
bing_api_key = os.getenv("BING_SUBSCRIPTION_KEY")
if not bing_api_key:
    print({"error": "BING_SUBSCRIPTION_KEY not found in environment variables"})
    
    # Validate API key format (Bing keys are typically 32 hex characters)
if len(bing_api_key) != 32:
    print({"error": f"Invalid Bing API key format. Expected 32 characters, got {len(bing_api_key)}"})

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "runner_agent"
os.environ["BING_SEARCH_URL"] = "https://api.bing.microsoft.com/v7.0/search"

LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Tavily tools
profile_search = TavilySearch(max_results=5, search_depth="basic", include_raw_content="text", exclude_domains=["tfrrs.org","athletic.net", "wikipedia.org"])
#swimcloud_search = TavilySearch(max_results=5, search_depth="advanced", include_raw_content="text", chunks_per_source = 5, country = "united states", exclude_domains=["tfrrs.org","athletic.net", "milesplit.com"])
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
extract = TavilyExtract(extract_depth="advanced")
#tools = [search, extract]

reasoning_llm = ChatOpenAI(model="o4-mini-2025-04-16", api_key=os.getenv("OPENAI_API_KEY"))
llm = ChatOpenAI(model="gpt-4.1-nano-2025-04-14", api_key=os.getenv("OPENAI_API_KEY"))

class RunnerProfileState(TypedDict):
    # Inputs
    runner_name: str
    college_name: str
    search_results: NotRequired[list[str]]
    page_snippets: NotRequired[list[dict]]      # {url, snippet}
    snippet_text: NotRequired[str]  # snippet text for selected profile URL
    selected_profile_url: NotRequired[str]
    full_profile_text: NotRequired[str]
    hometown: NotRequired[str]
    high_school: NotRequired[str]
    swim_background: NotRequired[str]
    next: NotRequired[str]  # next step in the workflow

    swim_search_results: NotRequired[list[dict]]  # raw hits: {url, title, content, score}
    swim_candidates:   NotRequired[list[str]]     # URLs that passed filter
    swim_profile_url:  NotRequired[str]           # top candidate (or null)
    swim_profile_text: NotRequired[str]           # full extracted page text

    has_swim_background:  NotRequired[bool]
    rationale:            NotRequired[str]
    match_confidence:     NotRequired[Literal["None", "Need Human Review"]]

    # Errors
    error: NotRequired[str]

{'error': 'BING_SUBSCRIPTION_KEY not found in environment variables'}
{'error': 'Invalid Bing API key format. Expected 32 characters, got 0'}


In [5]:
import json
from langchain.prompts import ChatPromptTemplate

# 1. search_pages
def search_pages(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    query = f"{state['runner_name']} {state['college_name']} track"
    res = profile_search.invoke(query)
    urls = [hit["url"] for hit in res.get("results", [])]
    # Add raw_content to each snippet dict
    snippets = [
        {
            "url": hit["url"],
            "snippet": hit.get("content", "")[:4000],
            "raw_content": hit.get("raw_content", "")
        }
        for hit in res.get("results", [])
        if hit.get("content") or hit.get("raw_content")
    ]
    urls = [s["url"] for s in snippets]
    return {
        "search_results": urls,
        "page_snippets": snippets
    }

# 2. select_profile_page
select_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are looking for the official college roster/profile page. "
     "Given these URL+snippet pairs for {runner_name} at {college_name}, "
     "pick the one that is the athlete’s roster page on the college site. "
     "It should be the INDIVIDUAL athlete's page, NOT a team roster page."
     "To help isolate the roster vs individual page, review the url format, it will often look like /college/roster/(athlete name or id)"
     "Return just the URL or null if none. Null should only be returned if certain the page is not related to the athlete."),
    ("human", "Pages:\n{pages}")
])

def select_profile_page(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    pages = state.get("page_snippets", [])
    if not pages:
        return {"error": "No pages to choose from."}
    blurb = "\n".join(
        f"[{i}] {p['url']}\n{p['snippet']}\n"
        for i, p in enumerate(pages, start=1)
    )
    prompt_input = select_prompt.invoke({
        "pages": blurb,
        "runner_name": state["runner_name"],
        "college_name": state["college_name"]
    })
    response = llm.invoke(prompt_input)
    chosen = response.content.strip()
    selected_url = chosen if chosen != "null" else None
    # Find raw_content for the selected URL
    snippet_text = None
    if selected_url:
        for p in pages:
            if p["url"] == selected_url:
                snippet_text = p.get("snippet", "")
                break
    return {
        "selected_profile_url": selected_url,
        "snippet_text": snippet_text
    }

#3. fetch_full_profile
def fetch_full_profile(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    """
    Fetch and concatenate extracted text from the top N (default 5) search result URLs.
    Returns a dict with the combined text under 'full_profile_text'.
    """
    # Get all URLs from search results
    urls: list[str] = state.get("search_results", [])
    if not urls:
        return {"error": "No search result URLs to extract."}

    # Extract content from all URLs
    res = extract.invoke({"urls": urls})
    if isinstance(res, str):
        if not res.strip():
            return {"error": "No content returned from extract.invoke."}
        try:
            res = json.loads(res)
        except json.JSONDecodeError as e:
            return {"error": f"Could not parse JSON from extract.invoke: {e}"}
    if not res or "results" not in res or not res["results"]:
        return {"error": "No results found in extraction response."}

    # Concatenate all extracted texts
    texts = []
    for result in res["results"]:
        text = result.get("raw_content") or result.get("content") or ""
        if text:
            texts.append(text)
    full_profile_text = "\n\n".join(texts)

    return {"full_profile_text": full_profile_text}
    
    """url = state.get("selected_profile_url")
    if not url:
        return {"error": "No profile URL selected."}
    res = extract.invoke({"urls": [url]})
    if isinstance(res, str):
        if not res.strip():
            return {"error": "No content returned from extract.invoke."}
        try:
            res = json.loads(res)
        except json.JSONDecodeError as e:
            return {"error": f"Could not parse JSON from extract.invoke: {e}"}
    if not res or "results" not in res or not res["results"]:
        return {"error": "No results found in extraction response."}

    first = res["results"][0]
    full = first.get("raw_content") or first.get("content") or ""
    return {"full_profile_text": full}"""

    '''url = state.get("selected_profile_url")
    if not url:
        return {"error": "No profile URL for Selenium fetch."}
    from selenium.webdriver import Chrome, ChromeOptions
    from bs4 import BeautifulSoup
    import time
    options = ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    browser = Chrome(options=options)
    browser.get(url)
    time.sleep(2)
    html = browser.page_source
    browser.quit()
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator="\n")
    return {"full_profile_text": text}
'''
    
# 4. Parse metadata and store into variables for searching swim background 
parse_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert data extractor. "
     "Given the full text of an athlete’s official roster page, "
     "extract hometown, high school, and any swim experience or interest. "
     "Highschool should be a short name, and not a list of accomplishments from high school. "
     "Return JSON with keys hometown, high_school, swim_background (short sentence or null). "
     "For swim_background, return 'yes' if there is any mention of swimming experience or interest related to the individual,"
     "If the swimming information is related to the college team program or other general world swimming and is not related to the speciif college runner, return 'no'."),
    ("human", "{profile_text}")
])

def safe_json_parse(text: str) -> dict:
    """
    Safely parse a JSON string, returning a dict.
    If parsing fails, returns {'error': 'Could not parse JSON'}
    """
    import re
    # Remove triple backticks and optional 'json' label
    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*|^```", "", cleaned)
    cleaned = re.sub(r"```$", "", cleaned)
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except Exception as e:
        return {"error": f"Could not parse JSON: {e}"}

def parse_metadata(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    full_text = state.get("full_profile_text", "")
    snippet_text = state.get("snippet_text", "")
    context_text = f"{full_text}\n{snippet_text}".strip()
    if not context_text:
        return {"error": "No full page or snippet text to parse."}
    prompt_input = parse_prompt.invoke({"profile_text": context_text})
    response = llm.invoke(prompt_input)
    data = safe_json_parse(response.content)
    if "error" in data:
        return data
    return {
        "hometown": data.get("hometown"),
        "high_school": data.get("high_school"),
        "swim_background": data.get("swim_background")
    }

In [6]:
def fetch_with_selenium(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    """
    Headless Selenium + BeautifulSoup fallback to fetch full profile text.
    Catches and returns errors as part of the output dict.
    """
    url = state.get("selected_profile_url")
    if not url:
        return {"error": "No profile URL for Selenium fetch."}
    from selenium.webdriver import Chrome, ChromeOptions
    from bs4 import BeautifulSoup
    import time
    browser = None
    try:
        options = ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("--disable-gpu")
        browser = Chrome(options=options)
        browser.get(url)
        time.sleep(2)
        html = browser.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(separator="\n")
        return {"full_profile_text": text}
    except Exception as e:
        return

def parse_metadata_selenium(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    """
    Parse metadata from full profile text using Selenium fallback.
    """
    full_text = state.get("full_profile_text", "")
    if not full_text:
        return {"error": "No full profile text to parse."}
    
    # Use the same parsing logic as before
    prompt_input = parse_prompt.invoke({"profile_text": full_text})
    response = llm.invoke(prompt_input)
    data = safe_json_parse(response.content)
    
    if "error" in data:
        return data
    
    return {
        "hometown": data.get("hometown"),
        "high_school": data.get("high_school"),
        "swim_background": data.get("swim_background")
    }

def route_condition(state: RunnerProfileState, config: Any) -> Literal["Pass", "Fail"]:
    hometown = state.get("hometown", "")
    if not hometown:
        return "Fail"
    return "Pass"

In [7]:
# 5. Search for potential swim profiles. 

'''
def search_swimcloud(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    """
    Search SwimCloud using SerpAPI Bing engine.
    Returns a dict with swim_search_results, swim_candidates, swim_profile_url, swim_profile_text.
    """
    from serpapi import GoogleSearch
    from selenium.webdriver import Chrome, ChromeOptions
    from bs4 import BeautifulSoup
    import time
   
    
    # Build search query with runner demographics
    search_query: str = f"{state['runner_name']} swimcloud {state.get('hometown', '')}"
    
    # Configure SerpAPI parameters
    params = {
    "engine": "bing",
    "q": search_query,
    "cc": "US",
    "google_domain": "google.com",
    "hl": "en",
    "gl": "us",
    "location": "United States",
    "api_key": "4df0b24ca2720db8006169295767b2e22c0dc02bdab977e96324b33a0f3b1b47"
    }
        
    try:
        search = GoogleSearch(params)
        search_response = search.get_dict()    
    # Validate response structure
        if "organic_results" not in search_response:
            return {"error": "No organic results found in SerpAPI response"}
        organic_results: list = search_response["organic_results"]
    except Exception as e:
        return {"error": f"SerpAPI Bing search error: {str(e)}"}    
           
    search_hits: list[Dict[str, Any]] = [
        {
            "url": result.get("link", ""),
            "title": result.get("title", ""),
            "content": result.get("snippet", ""),
            "raw_content": result.get("snippet", "")
        }
        for result in organic_results
    ]

    # Filter for SwimCloud URLs only - following project URL filtering pattern
    swimcloud_candidates: list[str] = [
        hit["url"] for hit in search_hits if hit["url"] and "swimcloud.com" in hit["url"]
    ][:3]
    #top_candidate: str | None = swimcloud_candidates[0] if swimcloud_candidates else None
    
    # Extract text from top candidate
    swim_profile_texts: list[str] = []
    for url in swimcloud_candidates:
        text: str = ""
        try:
            options = ChromeOptions()
            options.add_argument("--headless")
            options.add_argument("--disable-gpu")
            options.add_argument("--no-sandbox")
            options.add_argument("--disable-dev-shm-usage")
            options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
            browser = Chrome(options=options)
            browser.get(url)
            time.sleep(1)  # Polite scraping
            html_content: str = browser.page_source
            browser.quit()
            soup = BeautifulSoup(html_content, "html.parser")
            text = soup.get_text(separator="\n", strip=True)
        except Exception as selenium_error:
            for hit in search_hits:
                if hit["url"] == url:
                    text = hit.get("raw_content", "")
                    text += f"\n\n[Selenium extraction failed: {selenium_error}]"
                    break
        swim_profile_texts.append(text)

    swim_profile_text: str | None = "\n\n---\n\n".join(swim_profile_texts) if swim_profile_texts else None
    top_candidate: str | None = swimcloud_candidates[0] if swimcloud_candidates else None

    return {
        "swim_search_results": search_hits,
        "swim_candidates": swimcloud_candidates,
        "swim_profile_url": top_candidate,
        "swim_profile_text": swim_profile_text
    }
'''

# ...existing code...

'''def search_swimcloud(state: RunnerProfileState, config: Any) -> dict:
    """
    Search SwimCloud for runner by name and hometown.
    Use TavilySearch to get candidate URLs, then TavilyExtract to get full raw_content.
    Returns swim_search_results, swim_candidates, swim_profile_urls, and a single swim_profile_text.
    """
    name: str = state["runner_name"]
    hometown: str = state.get("hometown", "")
    query: str = f"swimcloud {name} {hometown}"

    # Step 1: Search for SwimCloud profiles
    search_result = swimcloud_search.invoke(query)
    hits: list[dict] = [
        {
            "url": hit["url"],
            "title": hit.get("title", ""),
            "content": hit.get("content", ""),
            "score": hit.get("score", 0.0),
            "raw_content": hit.get("raw_content", "")
        }
        for hit in search_result.get("results", [])
    ]

    candidates: list[str] = [
        h["url"] for h in hits
        if h["score"] >= 0.1 and "swimcloud.com" in h["url"]
    ][:2]

    # Step 2: Use TavilyExtract to get full raw_content for each candidate
    swim_profile_texts: list[str] = []
    if candidates:
        extract_result = extract.invoke({"urls": candidates})
        if isinstance(extract_result, str):
            try:
                extract_result = json.loads(extract_result)
            except Exception as e:
                extract_result = {"results": []}
        for result in extract_result.get("results", []):
            text = result.get("raw_content") or result.get("content") or ""
            if text:
                swim_profile_texts.append(text)
    swim_profile_text: str = "\n\n---\n\n".join(swim_profile_texts) if swim_profile_texts else ""

    return {
        "swim_search_results": hits,
        "swim_candidates": candidates,
        "swim_profile_urls": candidates,
        "swim_profile_text": swim_profile_text
    }'''

def search_swimcloud(state: RunnerProfileState, config: Any) -> dict:
    """
    Search SwimCloud for runner by name and hometown using two queries.
    Use TavilySearch to get candidate URLs, then TavilyExtract to get full raw_content.
    Returns swim_search_results, swim_candidates, swim_profile_urls, and a single swim_profile_text.
    """
    import json

    runner_name: str = state["runner_name"]
    hometown: str = state.get("hometown", "")
    queries: list[str] = [
        f"swimcloud {runner_name}",
        f"swimcloud {runner_name} {hometown}"
    ]

    all_hits: list[dict] = []
    all_candidates: set[str] = set()

    for query in queries:
        search_result = client.search(query = query, 
                                    search_depth="advanced",
                                    include_raw_content="text",
                                    chunks_per_source=5,
                                    country="united states",
                                    include_domains=["swimcloud.com"])
        if isinstance(search_result, str):
            try:
                search_result = json.loads(search_result)
            except Exception:
                search_result = {"results": []}
        hits: list[dict] = [
            {
                "url": hit.get("url", ""),
                "title": hit.get("title", ""),
                "content": hit.get("content", ""),
                "score": hit.get("score", 0.0),
                "raw_content": hit.get("raw_content", "")
            }
            for hit in search_result.get("results", [])
        ]
        all_hits.extend(hits)
    
    top_hits: list[dict] = sorted(
        all_hits, key=lambda h: h["score"], reverse=True
    )[:2]
    top_urls: list[str] = [h["url"] for h in top_hits]

    # Extract text from the top two URLs
    swim_profile_texts: list[str] = []
    if top_urls:
        extract_result = extract.invoke({"urls": top_urls})
        if isinstance(extract_result, str):
            try:
                extract_result = json.loads(extract_result)
            except Exception:
                extract_result = {"results": []}
        for result in extract_result.get("results", []):
            text = result.get("raw_content") or result.get("content") or ""
            if text:
                swim_profile_texts.append(text)
    swim_profile_text: str = "\n\n---\n\n".join(swim_profile_texts) if swim_profile_texts else ""

    return {
        "swim_search_results": all_hits,
        "swim_candidates": top_urls,
        "swim_profile_urls": top_urls,
        "swim_profile_text": swim_profile_text
    }

#6. If there was a swim profile, reason about whether the runner has a swimming background

reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert triathlon talent scout. "
     "Your task is to determine whether a collegiate runner may hava a background or interest in swimming."
     "You will be given the runner’s college profile page text and a potential matching SwimCloud profile page text,"
     "The Swimcloud profile page may match the runner's name, but it may not be the same person. "
     "The swimcloud profile may be from a young age, may differ slightly in name, but may still be relevant and show that this runner previously swam"
     "Especially if the swimcloud profile matches the runner's hometown, or in close proximity, you SHOULD consider the two people likely to be the same."
     "Place a strong emphasis that the runner likely matches the Swimcloud profile if the name is similar especially last name, given potential first name nicknames."
     "Your task is not to determine if the runner is a good swimmer, but rather to determine if they have ANY previous swimming experience or interest."
     "Use the context at hand to determine whether the runner has a swimming background or serious interest."
     "If a match between the runner and the SwimCloud profile is hard to determine, which would be when their is some direct matching information like hometown state or name,"
     "set match_confidence to 'Need Human Review', else when multiple criteria differs set to 'None'. "
     ""
     "Return a JSON object with keys:\n"
     "  has_swim_background (boolean),\n"
     "  rationale (string) — one or two sentences summarizing the evidence."
     "  match_confidence (string) — 'None' if confident, 'Need Human Review' if not."),
    ("human",
     "Runner Profile Page:\n```\n{profile_text}\n```\n\n"
     "SwimCloud Profile Page:\n```\n{swim_text}\n```")
])

def reason_about_swim_background(state: RunnerProfileState, config: Any) -> dict:
    profile = state.get("full_profile_text", "")
    swim = state.get("swim_profile_text", "")
    swim_interest = state.get("swim_background", "")
    if not profile or not swim:
        return {
            "has_swim_background": False,
            "rationale": "No profile or swim page text available.",
            "match_confidence": "None"
        }
    if swim_interest == "yes":
        # If we already have swim interest, return it directly
        return {
            "has_swim_background": True,
            "rationale": "Runner profile had mention of swimming.",
            "match_confidence": "None"
        }

    # Modern prompt invocation pattern
    prompt_input = reasoning_prompt.invoke({
        "profile_text": profile[:20000],
        "swim_text": swim[:18000]
    })
    response = reasoning_llm.invoke(prompt_input)
    llm_output = response.content.strip()

    # Parse the JSON output
    try:
        result = safe_json_parse(llm_output)
    except json.JSONDecodeError:
        # Fallback: assume false if the model garbled
        return {
            "has_swim_background": False,
            "rationale": "Could not parse LLM response.",
            "match_confidence": "Need Human Review"
        }

    return {
        "has_swim_background": result.get("has_swim_background", False),
        "rationale": result.get("rationale", ""),
        "match_confidence": result.get("match_confidence", "Need Human Review")
    }

In [8]:
from langgraph.graph import StateGraph,START
from IPython.display import display, Image

builder = StateGraph(RunnerProfileState)
builder.add_node("search",    search_pages)
builder.add_node("select",    select_profile_page)
builder.add_node("fetch",     fetch_full_profile)
builder.add_node("parse",     parse_metadata)
#builder.add_node("route", route_condition)
builder.add_node("fetch_with_selenium", fetch_with_selenium)
builder.add_node("parse_selenium", parse_metadata_selenium)
builder.add_node("search_swimcloud", search_swimcloud)
builder.add_node("reasoning", reason_about_swim_background)

builder.add_edge(START, "search")
builder.add_edge("search", "select")
builder.add_edge("select", "fetch")
builder.add_edge("fetch", "parse")
#builder.add_edge("parse", "check_demographics")
'''builder.add_conditional_edges("check_demographics", check_runner_demographics, {
        "fetch_with_selenium": "fetch_with_selenium",
        "search_swimcloud": "search_swimcloud",
    },)'''
builder.add_conditional_edges(
    source="parse",
    path=route_condition,
    path_map={"Fail": "fetch_with_selenium", "Pass": "search_swimcloud"},
)
builder.add_edge("fetch_with_selenium", "parse_selenium")
builder.add_edge("parse_selenium", "search_swimcloud")
builder.add_edge("search_swimcloud", "reasoning")

builder.set_finish_point("reasoning")
graph = builder.compile()
#display(Image(graph.get_graph(xray=True).draw_mermaid_png()))


In [9]:
import sys
from pathlib import Path
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
sys.path.append(str(Path(os.getcwd()).parent))
from db.db_connection import get_db_session
from db.models import Runner

#Import excel spreadsheet to prioritize matching runners
#SPREADSHEET_PATH = "C:\\Users\\jhigh\\Projects\\tri-recruiting\\etl\\data\\Recruitment Spreadsheet.xlsx"
SPREADSHEET_PATH ="C:\\Projects\\tri-recruiting\\etl\\data\\Recruitment Spreadsheet.xlsx"

#SPREADSHEET_PATH = ".\etl\data\RecruitmentSpreadsheet.xlsx"
dfs: dict[str, pd.DataFrame] = pd.read_excel(SPREADSHEET_PATH, sheet_name=None)

df_men = dfs['Men']
# Make first_name and last_name columns lowercase for matching
df_men['first_name'] = df_men['first_name'].str.lower()
df_men['last_name'] = df_men['last_name'].str.lower()
df_women = dfs['Women']
df_women['first_name'] = df_women['first_name'].str.lower()
df_women['last_name'] = df_women['last_name'].str.lower()

def get_next_runner():
    session = get_db_session()

    runner = session.query(Runner).filter(
            Runner.excel_match == 1,
            Runner.swimmer.is_(None),
            Runner.excel_swimmer.in_([0,1])
        ).first()
    #next swimmer 
    #runner = session.query(Runner).filter(Runner.swimmer == None).first()
    first = runner.first_name
    last = runner.last_name
    runner_name = f"{first} {last}"
    college = runner.college_team
    excel_swimmer = runner.excel_swimmer
    session.close()
    
    return runner_name, college, excel_swimmer

def save_runner_profile(result: dict) -> None:
    """
    Save runner profile information to the database.
    Uses SQLAlchemy session and upsert logic.
    """
    session = get_db_session()
    try:
        runner_name = result.get("runner_name", "").strip().lower()
        name_parts = runner_name.split()
        if len(name_parts) < 2:
            print(f"Invalid runner_name format: '{runner_name}'")
            return
        first = name_parts[0]
        last = " ".join(name_parts[1:])  # Support multi-part last names
        if len(name_parts) > 2 and name_parts[1].startswith("o") and name_parts[2].startswith("'"):
            last = f"{name_parts[1]}{name_parts[2]}"
            if len(name_parts) > 3:
                last += " " + " ".join(name_parts[3:])

        runner = session.query(Runner).filter_by(first_name=first, last_name=last).first()

        if not runner:
            print("Runner not found in database")

        runner.hometown = result.get("hometown")
        runner.high_school = result.get("high_school")
        runner.swim_background = result.get("swim_background")
        runner.runner_url = result.get("selected_profile_url")
        runner.runner_text = result.get("full_profile_text", "")
        runner.swimmer = result.get("has_swim_background", None)
        runner.swim_url = result.get("swim_profile_url")
        runner.swim_text = result.get("swim_profile_text", "")
        runner.rationale = result.get("rationale", "")
        runner.match_confidence = result.get("match_confidence", "Need Human Review")

        session.commit()
        #print(f"Saved runner profile for {result.get('runner_name')}")
    except SQLAlchemyError as e:
        print(f"Database error: {e}")
        session.rollback()
    finally:
        session.close()

Python-dotenv could not parse statement starting at line 1


In [10]:
runner_name, college, excel_swimmer = get_next_runner()
print(f"Updating runner: {runner_name}, {college}")

Updating runner: berlyn schutz, Nebraska


In [ ]:
init = RunnerProfileState(runner_name=runner_name, college_name=college)
result = graph.invoke(init)

# Print only the requested fields
print("Runner Name:", result.get("runner_name"))
print("Profile URL:", result.get("selected_profile_url"))
print("Hometown:", result.get("hometown"))
print("High School:", result.get("high_school"))
print("Swimming Experience:", result.get("swim_background"))
print("Swimming Background:", result.get("has_swim_background"))
print("Rationale:", result.get("rationale", "No rationale provided."))
print("Match Confidence:", result.get("match_confidence", "Need Human Review"))

save_runner_profile(result)

if excel_swimmer is not None: 
    if result.get('has_swim_background') != excel_swimmer:
        print("WARNING")
    else:
        print("Excel swimmer matches the result.")

In [12]:
for i in range(20):
    runner_name, college, excel_swimmer = get_next_runner()
    print(f"Updating runner: {runner_name}, {college}")
    init = RunnerProfileState(runner_name=runner_name, college_name=college)
    result = graph.invoke(init)
    save_runner_profile(result)

    if excel_swimmer is not None: 
        if result.get('has_swim_background') != excel_swimmer:
            print("WARNING")
        #else:
            #print("Excel swimmer matches the result.")

Updating runner: silvia jelelgo, Clemson
Updating runner: melissa riggins, Georgetown
Updating runner: tatiana cornejo, Cal Poly
Updating runner: julia rosenberg, Vanderbilt
Updating runner: erin vringer, Utah
WARNING
Updating runner: trixie wraith, Bradley
Updating runner: kileigh kane, Penn State
Updating runner: reagan baesler, North Dakota State
Updating runner: brooke garter, Belmont
Updating runner: leoni mierswa, SMU
Updating runner: katie turk, Maryland
Updating runner: anneken viljoen, Missouri
Updating runner: gretchen farley, Notre Dame
Updating runner: kaitlyn sheppard, Bradley
Updating runner: ally kruger, Kentucky
Updating runner: lily myers, Indiana
Updating runner: hanna bruckmayer, New Mexico
Updating runner: lea hatcher, Penn State
Updating runner: isatu n'diaye, UC Riverside
Updating runner: grace link, North Dakota State


In [ ]:
def spreadsheet_has_runner(df: pd.DataFrame, first_name: str, last_name: str, college: str) -> str | None:
    """
    Return the swimmer column value if runner is found in df by name and college, else None.
    """
    matches = df[
        (df['first_name'] == first_name) &
        (df['last_name'] == last_name) &
        ((df.get('college', df.get('college_team', '')) == college))
    ]
    if not matches.empty:
        return matches.iloc[0].get('swimmer', None)
    return None

session = get_db_session()
matching_results = []

runners = session.query(Runner).all()
for runner in runners:
    first_name = runner.first_name.lower()
    last_name = runner.last_name.lower()
    college = runner.college_team
    swimmer_val_women = spreadsheet_has_runner(df_women, first_name, last_name, college)
    swimmer_val_men = spreadsheet_has_runner(df_men, first_name, last_name, college)

    swimmer_val = swimmer_val_women if swimmer_val_women is not None else swimmer_val_men
    if swimmer_val is not None:
        matching_results.append({
            "first_name": first_name,
            "last_name": last_name,
            "college": college,
            "swimmer": swimmer_val
        })
session.close()

In [ ]:
session = get_db_session()
try:
    for result in matching_results:
        first_name = result["first_name"]
        last_name = result["last_name"]
        college = result["college"]
        swimmer_val = result["swimmer"]

        runner = session.query(Runner).filter_by(
            first_name=first_name,
            last_name=last_name,
            college_team=college
        ).first()

        if runner:
            runner.excel_match = True
            if swimmer_val == 1:
                runner.excel_swimmer = True
            elif swimmer_val == 0:
                runner.excel_swimmer = False
            elif swimmer_val == -1:
                runner.excel_swimmer = None
    session.commit()
    print("excel_swimmer column updated for matching runners.")
except SQLAlchemyError as e:
    print(f"Database error: {e}")
    session.rollback()
finally:
    session.close()

In [ ]:
import pandas as pd
from sqlalchemy.orm import Session
from db.db_connection import get_db_session
from db.models import Runner

SPREADSHEET_PATH = "C:\Users\jhigh\Projects\tri-recruiting\etl\data\Recruitment Spreadsheet.xlsx"

def load_spreadsheet(path: str) -> pd.DataFrame:
    """Load recruitment spreadsheet into a DataFrame."""
    return pd.read_excel(path)

def find_runner_in_db(session: Session, first_name: str, last_name: str, college: str) -> Runner | None:
    """Find a runner in the database by name and college."""
    return session.query(Runner).filter_by(
        first_name=first_name.lower(),
        last_name=last_name.lower(),
        college_team=college
    ).first()

def cross_reference_spreadsheet() -> None:
    """Cross-reference spreadsheet data with database records and print results."""
    df = load_spreadsheet(SPREADSHEET_PATH)
    session = get_db_session()
    matches = []
    non_matches = []

    for _, row in df.iterrows():
        first_name = str(row.get("First Name", "")).strip()
        last_name = str(row.get("Last Name", "")).strip()
        college = str(row.get("College", "")).strip()

        runner = find_runner_in_db(session, first_name, last_name, college)
        if runner:
            matches.append((first_name, last_name, college))
        else:
            non_matches.append((first_name, last_name, college))

    session.close()

    print("Matched runners:")
    for m in matches:
        print(f"{m[0]} {m[1]} ({m[2]})")

    print("\nUnmatched runners:")
    for nm in non_matches:
        print(f"{nm[0]} {nm[1]} ({nm[2]})")



In [ ]:
#Old functions to save for reference
# 1. search_pages
def search_pages(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    query = f"{state['runner_name']} {state['college_name']} roster profile"
    res = search.invoke(query)
    urls = [hit["url"] for hit in res.get("results", [])]
    snippets = [
        {"url": hit["url"], "snippet": hit.get("content", "")[:2000]}
        for hit in res.get("results", [])
    ]
    return {
        "search_results": urls,
        "page_snippets": snippets
    }